# Laboratorio 9 - Deep Learning

- Juan Solís - 23720
- Victor Pérez - 23731
- Diego Flores - 23714
- Nils Muralles - 23727
- Isabella Recinos - 23003

## Task 4

### Task 4.1

Se reconstruyen el VAE con β = 1 (Task 1) y la U-Net condicional (Task 3) con la misma semilla e hiperparámetros. Si existen los checkpoints en `data/` se cargan; si no, se entrenan.

In [ ]:
import math, time
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path("data").mkdir(exist_ok=True)

data = datasets.FashionMNIST("data", train=True, download=True)
X01 = data.data.float() / 255.0          # [0,1] para el VAE
Y = data.targets
print("semilla:", SEED, "| device:", device)

**VAE β = 1** (misma arquitectura y pérdida de la Task 1.2).

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(),
                                 nn.Linear(512, 784), nn.Sigmoid())

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # z = mu + sigma * eps
        return self.dec(z), mu, logvar

x_train_vae = X01[:-5000].view(-1, 784).to(device)
vae = VAE().to(device)
vae_ckpt = Path("data/task1_vae_beta1.pt")
if vae_ckpt.exists():
    vae.load_state_dict(torch.load(vae_ckpt, map_location=device))
else:
    torch.manual_seed(SEED)
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
    for ep in range(20):
        perm = torch.randperm(len(x_train_vae), device=device)
        for i in range(0, len(x_train_vae), 128):
            xb = x_train_vae[perm[i:i + 128]]
            x_hat, mu, logvar = vae(xb)
            # L_beta = ||x - x_hat||^2 + beta * KL, con beta = 1
            kl = 0.5 * torch.sum(mu**2 + logvar.exp() - 1 - logvar, dim=1)
            loss = (torch.sum((xb - x_hat) ** 2, dim=1) + kl).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    torch.save(vae.state_dict(), vae_ckpt)
vae.eval()
print("VAE listo")

**U-Net condicional y calendario lineal** (Tasks 2.1, 3.1 y 3.2).

In [ ]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim debe ser par")
        frequencies = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float32)
            * (-math.log(10000.0) / dim)
        )
        self.dim = dim
        self.register_buffer("frequencies", frequencies)

    def forward(self, t):
        angles = t.float().unsqueeze(1) * self.frequencies.unsqueeze(0)
        embedding = torch.empty(t.size(0), self.dim, device=t.device)
        embedding[:, 0::2] = torch.sin(angles)
        embedding[:, 1::2] = torch.cos(angles)
        return embedding


class ConditionedBlock(nn.Module):
    def __init__(self, in_channels, out_channels, condition_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.condition_projection = nn.Linear(condition_dim, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.activation = nn.SiLU()

    def forward(self, x, condition):
        h = self.activation(self.norm1(self.conv1(x)))
        h = h + self.condition_projection(condition)[:, :, None, None]
        h = self.activation(self.norm2(self.conv2(h)))
        return h


class SmallConditionalUNet(nn.Module):
    def __init__(self, condition_dim=64):
        super().__init__()
        self.time_embedding = SinusoidalEmbedding(condition_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.SiLU(),
            nn.Linear(condition_dim, condition_dim),
        )
        self.class_embedding = nn.Embedding(11, condition_dim)

        self.encoder_28 = ConditionedBlock(1, 32, condition_dim)
        self.downsample = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.encoder_14 = ConditionedBlock(64, 64, condition_dim)
        self.bottleneck = ConditionedBlock(64, 64, condition_dim)

        self.upsample = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.decoder_28 = ConditionedBlock(64, 32, condition_dim)
        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x, t, c):
        time_condition = self.time_mlp(self.time_embedding(t))
        class_condition = self.class_embedding(c)
        condition = time_condition + class_condition

        skip = self.encoder_28(x, condition)
        h = self.downsample(skip)
        h = self.encoder_14(h, condition)
        h = self.bottleneck(h, condition)
        h = self.upsample(h)
        h = torch.cat([h, skip], dim=1)
        h = self.decoder_28(h, condition)
        return self.output(h)



T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)    # calendario lineal beta_t
alphas = 1 - betas                                       # alpha_t = 1 - beta_t
alpha_bar = torch.cumprod(alphas, dim=0)                 # alpha_bar_t = prod alpha_s

X = (X01 * 2 - 1).unsqueeze(1)                           # difusión en [-1,1]
x_train, y_train = X[:-5000].to(device), Y[:-5000].to(device)

def q_sample(x0, t, eps):
    # x_t = sqrt(alpha_bar_t) x0 + sqrt(1 - alpha_bar_t) eps
    ab = alpha_bar[t - 1].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

torch.manual_seed(SEED)
unet = SmallConditionalUNet().to(device)
optimizer = torch.optim.Adam(unet.parameters(), lr=2e-4)
ckpt = Path("data/task3_diffusion_checkpoint.pt")
train_losses = []
if ckpt.exists():
    saved = torch.load(ckpt, map_location=device)
    unet.load_state_dict(saved["model"]); optimizer.load_state_dict(saved["optimizer"])
    train_losses = saved["train_losses"]

t0 = time.perf_counter()
for epoch in range(len(train_losses), 30):
    unet.train()
    perm = torch.randperm(len(x_train), device=device)
    total = 0.0
    for i in range(0, len(x_train), 128):
        idx = perm[i:i + 128]
        x0, c = x_train[idx], y_train[idx].clone()
        t = torch.randint(1, T + 1, (len(x0),), device=device)
        eps = torch.randn_like(x0)
        c[torch.rand(len(x0), device=device) < 0.1] = 10   # condición nula con p_uncond
        loss = torch.mean((eps - unet(q_sample(x0, t, eps), t, c)) ** 2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(x0)
    train_losses.append(total / len(x_train))
    torch.save({"model": unet.state_dict(), "optimizer": optimizer.state_dict(),
                "train_losses": train_losses}, ckpt)
unet.eval()
train_time = time.perf_counter() - t0
print(f"Pérdida final: {train_losses[-1]:.5f} | tiempo de entrenamiento en esta sesión: {train_time:.1f} s")

In [ ]:
@torch.no_grad()
def ancestral_sample(model, labels):
    # Muestreo ancestral de la Task 3.3 (w = 1: solo predicción condicional)
    x = torch.randn(len(labels), 1, 28, 28, device=device)   # x_T ~ N(0, I)
    for step in range(T, 0, -1):
        t = torch.full((len(labels),), step, device=device, dtype=torch.long)
        eps_hat = model(x, t, labels)
        # x_{t-1} = 1/sqrt(alpha_t) (x_t - (1-alpha_t)/sqrt(1-alpha_bar_t) eps_hat) + sigma_t z
        mean = (x - (1 - alphas[step - 1]) / (1 - alpha_bar[step - 1]).sqrt() * eps_hat) / alphas[step - 1].sqrt()
        x = mean + betas[step - 1].sqrt() * torch.randn_like(x) if step > 1 else mean
    return x.clamp(-1, 1)

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

# 500 muestras con w = 1 (50 por clase, como en la Task 3.4)
samples_ckpt = Path("data/task3_guided_w1.pt")
if samples_ckpt.exists():
    diff_500 = torch.load(samples_ckpt, map_location=device)["samples"].to(device)
else:
    torch.manual_seed(SEED)
    diff_500 = ancestral_sample(unet, torch.arange(10, device=device).repeat_interleave(50))
print(diff_500.shape)

**a.** Nitidez (Task 1.4 d) sobre 500 imágenes de cada fuente. Las muestras de difusión se pasan de [−1,1] a [0,1] para que la escala sea la misma que la del VAE y las reales.

In [ ]:
def sharpness(imgs):
    # Magnitud media de la diferencia entre píxeles horizontalmente adyacentes
    imgs = imgs.reshape(-1, 28, 28)
    return (imgs[:, :, 1:] - imgs[:, :, :-1]).abs().mean().item()

torch.manual_seed(SEED)
with torch.no_grad():
    vae_500 = vae.dec(torch.randn(500, 2, device=device))   # z ~ N(0, I)
real_500 = X01[-5000:][:500]                                 # validación, [0,1]

sharp = {
    "Real": sharpness(real_500),
    "Difusión w=1": sharpness((diff_500 + 1) / 2),
    "VAE β=1": sharpness(vae_500),
}
for k, v in sharp.items():
    print(f"{k:14s} {v:.4f}")

**b.** Tiempo para generar 100 imágenes y número de evaluaciones de red.

In [ ]:
labels_100 = torch.arange(10, device=device).repeat_interleave(10)

# Calentamiento para no medir la inicialización de CUDA
with torch.no_grad():
    vae.dec(torch.randn(100, 2, device=device)); unet(torch.randn(100, 1, 28, 28, device=device), labels_100 + 1, labels_100)

torch.manual_seed(SEED); sync(); t0 = time.perf_counter()
with torch.no_grad():
    vae.dec(torch.randn(100, 2, device=device))          # 1 pasada del decoder
sync(); t_vae = time.perf_counter() - t0

torch.manual_seed(SEED); sync(); t0 = time.perf_counter()
ancestral_sample(unet, labels_100)                       # T = 1000 pasadas de la U-Net
sync(); t_diff = time.perf_counter() - t0

print(f"VAE:      {t_vae * 1000:.3f} ms | evaluaciones de red: 1")
print(f"Difusión: {t_diff:.3f} s    | evaluaciones de red: {T}")
print(f"Razón de tiempos difusión/VAE: {t_diff / t_vae:,.0f}x")

**c.** *(ver respuesta en la celda final)*

### Task 4.2

Estimador insesgado pass@k = 1 − C(n−c, k)/C(n, k) (vale 1 si n − c < k) y versión ingenua 1 − (1 − c/n)^k.

In [ ]:
import pandas as pd
from math import comb

n = 20
problems = {"P1": 0, "P2": 2, "P3": 5, "P4": 9, "P5": 14, "P6": 20}

def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)

def pass_naive(n, c, k):
    return 1 - (1 - c / n) ** k

tbl = pd.DataFrame({f"pass@{k}": {p: pass_at_k(n, c, k) for p, c in problems.items()} for k in (1, 5, 10)})
tbl["ingenua@5"] = {p: pass_naive(n, c, 5) for p, c in problems.items()}
tbl.loc["Promedio"] = tbl.mean()
print(tbl.round(4).to_string())

# b. P2: pass@k = 1 - (20-k)(19-k)/380; menor k con pass@k > 0.9
k_min = next(k for k in range(1, 21) if pass_at_k(n, 2, k) > 0.9)
assert all(abs(pass_at_k(n, 2, k) - (1 - (20 - k) * (19 - k) / 380)) < 1e-12 for k in range(1, 19))
print("Menor k con pass@k > 0.9 para P2:", k_min, "->", round(pass_at_k(n, 2, k_min), 4))

## Respuestas Task 4

_PENDIENTE_